# P2 Adaptive / Dual-Threshold Confidence Gate — Full Colab

**Project:** Agentic Explainable Agricultural Decision Support System using Multi-Agent AI, RAG and Machine Learning

This notebook is the next conference-paper experiment after P1. It uses the already-saved P1/R0/R1/R2 outputs from `MyDrive/Capstone_Conference_Paper/` and automatically runs:

- exact leakage-free split reconstruction
- confidence-estimator retraining
- P1 single-threshold reconstruction (validation only)
- **P2 adaptive dual-threshold search on validation only**
- untouched test comparison: Original vs P1 vs P2
- Qwen end-to-end Original RAG vs P1 vs P2
- checkpoint/resume
- CSV + PNG + automatic conference-paper result summary

### P2 policy
- **High confidence** → Primary evidence
- **Medium confidence** → Secondary evidence used only when more context is needed
- **Low confidence** → Rejected
- No usable evidence → Abstain

### Scientific rule
P2 thresholds are selected **only on the validation set**. The test set is not used for tuning.

### Run
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Allow Google Drive access when asked


In [ ]:
# 0. SAFE DEPENDENCY SETUP — do not upgrade NumPy/Pandas/scikit-learn
import sys, subprocess, importlib.util
required = {
    'transformers':'transformers>=4.51,<5',
    'accelerate':'accelerate',
    'sentence_transformers':'sentence-transformers',
}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Dependency check complete. Core scientific stack was not replaced.')


In [ ]:
# 1. CONFIG + IMPORTS + GPU + DRIVE
SEED=42
TOP_K=5
MIN_COVERAGE=0.90
MODEL_NAME='Qwen/Qwen3-4B-Instruct-2507'
BI_ENCODER_NAME='sentence-transformers/all-MiniLM-L6-v2'
MAX_CONTEXT_CHARS=1600
MAX_INPUT_TOKENS=4096
MAX_NEW_TOKENS=110
DRIVE_DIR='/content/drive/MyDrive/Capstone_Conference_Paper'
MAX_GENERATION_QUERIES=None  # None = all untouched test queries

import os,re,json,time,random,warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sentence_transformers import SentenceTransformer
from google.colab import drive
warnings.filterwarnings('ignore')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then Run all again.')
print('GPU:',torch.cuda.get_device_name(0))
drive.mount('/content/drive',force_remount=False)
SAVE_DIR=Path(DRIVE_DIR)
if not SAVE_DIR.exists(): raise RuntimeError('P1 result folder not found. Run Capstone_Conference_Paper_Full_Colab_FIXED.ipynb first.')
print('Using result folder:',SAVE_DIR)


## 2. Load P1 scored query–chunk data and reconstruct the exact split


In [ ]:
SCORED_PATH=SAVE_DIR/'R2_all_scored_query_chunk_pairs.csv'
if not SCORED_PATH.exists(): raise FileNotFoundError(f'Missing {SCORED_PATH}. Run the P1 full notebook first.')
eval_df=pd.read_csv(SCORED_PATH)
needed=['query','chunk','chunk_number','relevance_label','embedding_similarity','cross_encoder_score']
missing=[c for c in needed if c not in eval_df.columns]
if missing: raise ValueError(f'Missing required columns: {missing}')

gate_df=eval_df.copy()
gate_df['original_rank_signal']=1.0/gate_df['chunk_number']
gate_df['bi_rank']=gate_df.groupby('query')['embedding_similarity'].rank(method='first',ascending=False)
gate_df['cross_rank']=gate_df.groupby('query')['cross_encoder_score'].rank(method='first',ascending=False)
gate_df['bi_rank_signal']=1.0/gate_df['bi_rank']
gate_df['cross_rank_signal']=1.0/gate_df['cross_rank']
FEATURES=['original_rank_signal','embedding_similarity','cross_encoder_score','bi_rank_signal','cross_rank_signal']
TARGET='relevance_label'
gate_df=gate_df.dropna(subset=FEATURES+[TARGET]).copy()

s1=GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED)
train_idx,temp_idx=next(s1.split(gate_df,groups=gate_df['query']))
train_df=gate_df.iloc[train_idx].copy(); temp_df=gate_df.iloc[temp_idx].copy()
s2=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED)
val_idx,test_idx=next(s2.split(temp_df,groups=temp_df['query']))
val_df=temp_df.iloc[val_idx].copy(); test_df=temp_df.iloc[test_idx].copy()
a=set(train_df['query']); b=set(val_df['query']); c=set(test_df['query'])
assert not(a&b or a&c or b&c),'Query leakage detected.'
print('Train / Validation / Test queries:',train_df['query'].nunique(),val_df['query'].nunique(),test_df['query'].nunique())
print('Query overlap = 0 / 0 / 0')


## 3. Retrain confidence estimator and reconstruct P1 threshold using validation only


In [ ]:
gate_model=Pipeline([('scaler',StandardScaler()),('classifier',LogisticRegression(class_weight='balanced',max_iter=2000,random_state=SEED))])
gate_model.fit(train_df[FEATURES],train_df[TARGET].astype(int))
val_df=val_df.copy(); test_df=test_df.copy()
val_df['evidence_confidence']=gate_model.predict_proba(val_df[FEATURES])[:,1]
test_df['evidence_confidence']=gate_model.predict_proba(test_df[FEATURES])[:,1]

def single_threshold_metrics(df,threshold):
    rows=[]
    for q,g in df.groupby('query'):
        accepted=g[g['evidence_confidence']>=threshold].sort_values('evidence_confidence',ascending=False).head(TOP_K)
        covered=int(len(accepted)>0); total=g['relevance_label'].sum()
        rows.append({'query':q,'precision':accepted['relevance_label'].mean() if covered else np.nan,'recall':accepted['relevance_label'].sum()/total if total>0 and covered else 0.0,'top1':int(accepted.iloc[0]['relevance_label']) if covered else np.nan,'coverage':covered,'abstained':1-covered,'context_count':len(accepted)})
    z=pd.DataFrame(rows)
    return {'precision':z['precision'].mean(skipna=True),'recall':z['recall'].mean(),'top1':z['top1'].mean(skipna=True),'coverage':z['coverage'].mean(),'abstention':z['abstained'].mean(),'mean_context':z['context_count'].mean()},z

p1_grid=[]
for th in np.arange(.20,.91,.05):
    m,_=single_threshold_metrics(val_df,float(th)); p1_grid.append({'threshold':round(float(th),2),**m})
p1_grid=pd.DataFrame(p1_grid)
eligible=p1_grid[p1_grid['coverage']>=MIN_COVERAGE].copy()
P1_THRESHOLD=float(eligible.sort_values(['precision','top1'],ascending=False).iloc[0]['threshold'])
print('Reconstructed P1 threshold:',P1_THRESHOLD)
display(p1_grid.round(4))


## 4. P2 adaptive dual-threshold validation search

P2 uses a high threshold for primary evidence and a lower threshold for secondary fallback evidence. Primary evidence is always preferred; secondary evidence is added only until the target context size is reached.

The selected configuration must keep at least 90% query coverage and must improve validation precision over the original retrieval by at least 5 percentage points. Among eligible configurations, the one with the highest precision–recall F1 is selected.


In [ ]:
def original_metrics(df):
    rows=[]
    for q,g in df.groupby('query'):
        base=g.sort_values('chunk_number').head(TOP_K); total=g['relevance_label'].sum()
        rows.append({'query':q,'precision':base['relevance_label'].mean(),'recall':base['relevance_label'].sum()/total if total>0 else np.nan,'top1':int(base.iloc[0]['relevance_label']),'context_count':len(base)})
    z=pd.DataFrame(rows)
    return {'precision':z['precision'].mean(),'recall':z['recall'].mean(),'top1':z['top1'].mean(),'coverage':1.0,'abstention':0.0,'mean_context':z['context_count'].mean()},z

def adaptive_select(group,low,high,target_context):
    g=group.copy()
    primary=g[g['evidence_confidence']>=high].sort_values('evidence_confidence',ascending=False).copy()
    secondary=g[(g['evidence_confidence']>=low)&(g['evidence_confidence']<high)].sort_values('evidence_confidence',ascending=False).copy()
    primary['evidence_tier']='PRIMARY'; secondary['evidence_tier']='SECONDARY'
    chosen=primary.head(target_context).copy()
    remaining=max(0,target_context-len(chosen))
    if remaining>0: chosen=pd.concat([chosen,secondary.head(remaining)],ignore_index=False)
    return chosen

def adaptive_metrics(df,low,high,target_context):
    rows=[]
    for q,g in df.groupby('query'):
        selected=adaptive_select(g,low,high,target_context); covered=int(len(selected)>0); total=g['relevance_label'].sum()
        rows.append({'query':q,'precision':selected['relevance_label'].mean() if covered else np.nan,'recall':selected['relevance_label'].sum()/total if total>0 and covered else 0.0,'top1':int(selected.iloc[0]['relevance_label']) if covered else np.nan,'coverage':covered,'abstained':1-covered,'context_count':len(selected)})
    z=pd.DataFrame(rows)
    p=z['precision'].mean(skipna=True); r=z['recall'].mean(); f1=(2*p*r/(p+r)) if (p+r)>0 else 0.0
    return {'precision':p,'recall':r,'f1':f1,'top1':z['top1'].mean(skipna=True),'coverage':z['coverage'].mean(),'abstention':z['abstained'].mean(),'mean_context':z['context_count'].mean()},z

original_val,_=original_metrics(val_df)
p1_val,_=single_threshold_metrics(val_df,P1_THRESHOLD)
MIN_P2_PRECISION=original_val['precision']+0.05
search_rows=[]
for low in np.arange(.20,.56,.05):
    for high in np.arange(.45,.81,.05):
        if high<=low: continue
        for target in [2,3]:
            m,_=adaptive_metrics(val_df,float(low),float(high),target)
            search_rows.append({'low_threshold':round(float(low),2),'high_threshold':round(float(high),2),'target_context':target,**m})
p2_grid=pd.DataFrame(search_rows)
eligible=p2_grid[(p2_grid['coverage']>=MIN_COVERAGE)&(p2_grid['precision']>=MIN_P2_PRECISION)].copy()
if eligible.empty:
    eligible=p2_grid[p2_grid['coverage']>=MIN_COVERAGE].copy()
best=eligible.sort_values(['f1','precision','top1'],ascending=False).iloc[0]
P2_LOW=float(best['low_threshold']); P2_HIGH=float(best['high_threshold']); P2_TARGET=int(best['target_context'])
print('Original validation:',original_val)
print('P1 validation:',p1_val)
print('Selected P2:',{'low':P2_LOW,'high':P2_HIGH,'target_context':P2_TARGET})
display(best.to_frame(name='Validation Value'))
p2_grid.to_csv(SAVE_DIR/'P2_validation_grid.csv',index=False)


## 5. Untouched test comparison: Original vs P1 vs P2


In [ ]:
original_test,original_q=original_metrics(test_df)
p1_test,p1_q=single_threshold_metrics(test_df,P1_THRESHOLD)
p2_test,p2_q=adaptive_metrics(test_df,P2_LOW,P2_HIGH,P2_TARGET)

TEST_COMPARISON=pd.DataFrame({
 'Metric':['Evidence Precision','Evidence Recall','Top-1 Relevance','Query Coverage','Abstention Rate','Mean Context Count'],
 'Original Retrieval':[original_test['precision'],original_test['recall'],original_test['top1'],original_test['coverage'],original_test['abstention'],original_test['mean_context']],
 'P1 Single-Threshold':[p1_test['precision'],p1_test['recall'],p1_test['top1'],p1_test['coverage'],p1_test['abstention'],p1_test['mean_context']],
 'P2 Adaptive Gate':[p2_test['precision'],p2_test['recall'],p2_test['top1'],p2_test['coverage'],p2_test['abstention'],p2_test['mean_context']]
})
display(TEST_COMPARISON.round(4))
TEST_COMPARISON.to_csv(SAVE_DIR/'P2_Test_Evidence_Comparison.csv',index=False)
p2_q.to_csv(SAVE_DIR/'P2_Test_Query_Level_Results.csv',index=False)

# Store selected P2 context/tier per test query
selected_rows=[]
for q,g in test_df.groupby('query'):
    sel=adaptive_select(g,P2_LOW,P2_HIGH,P2_TARGET).copy()
    if len(sel)==0:
        selected_rows.append({'query':q,'chunk':None,'evidence_confidence':np.nan,'evidence_tier':'ABSTAIN','relevance_label':np.nan})
    else:
        for _,r in sel.iterrows():
            selected_rows.append({'query':q,'chunk':r['chunk'],'evidence_confidence':r['evidence_confidence'],'evidence_tier':r['evidence_tier'],'relevance_label':r['relevance_label']})
P2_SELECTED=pd.DataFrame(selected_rows)
P2_SELECTED.to_csv(SAVE_DIR/'P2_Selected_Test_Evidence.csv',index=False)


## 6. Evidence comparison figure


In [ ]:
plot=TEST_COMPARISON[TEST_COMPARISON['Metric'].isin(['Evidence Precision','Evidence Recall','Top-1 Relevance','Query Coverage'])].set_index('Metric')
ax=plot.plot(kind='bar',figsize=(10,5)); ax.set_ylim(0,1.02); ax.set_ylabel('Score'); ax.set_title('Original vs P1 vs P2 — Evidence Selection'); plt.xticks(rotation=10,ha='right'); plt.tight_layout(); plt.savefig(SAVE_DIR/'Figure_P2_Evidence_Comparison.png',dpi=220); plt.show()


# 7. End-to-End Qwen: G1 Original vs G2 P1 vs G3 P2

If the previous `G1_G2_Qwen_RAG_Full_Test.csv` exists, G1/G2 answers are reused and only P2 answers are generated. Otherwise all three systems are generated automatically.


In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
llm=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.float16,device_map='auto',low_cpu_mem_usage=True)
llm.eval()
print('Qwen device:',next(llm.parameters()).device)
if 'cuda' not in str(next(llm.parameters()).device): raise RuntimeError('Qwen did not load on GPU.')

SYSTEM_PROMPT='''You are an agricultural advisory assistant. Answer the farmer question using ONLY the supplied evidence. Do not invent facts. If evidence is insufficient, say: "Insufficient evidence to provide a reliable answer." Keep the answer concise and practical. Cite supporting evidence as [E1], [E2], etc. Treat evidence marked SECONDARY as lower-confidence support and do not rely on it when it conflicts with PRIMARY evidence.'''.strip()

def generate_answer(query,contexts):
    pieces=[]
    for i,item in enumerate(contexts):
        if isinstance(item,tuple): tier,text=item
        else: tier,text='EVIDENCE',item
        pieces.append(f'[E{i+1}] ({tier}) {str(text).strip()[:MAX_CONTEXT_CHARS]}')
    evidence='\n\n'.join(pieces)
    messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':f'Farmer Question:\n{query}\n\nAgricultural Evidence:\n{evidence}\n\nProvide an evidence-grounded answer.'}]
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True)
    inputs={k:v.to(llm.device) for k,v in inputs.items()}
    if inputs['input_ids'].shape[1]>MAX_INPUT_TOKENS:
        inputs['input_ids']=inputs['input_ids'][:,-MAX_INPUT_TOKENS:]
        if 'attention_mask' in inputs: inputs['attention_mask']=inputs['attention_mask'][:,-MAX_INPUT_TOKENS:]
    n=inputs['input_ids'].shape[1]; start=time.perf_counter()
    with torch.inference_mode(): out=llm.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,use_cache=True,pad_token_id=tokenizer.eos_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0,n:],skip_special_tokens=True).strip(),time.perf_counter()-start,n


In [ ]:
# 8. GENERATE / REUSE G1, G2, G3 WITH CHECKPOINT
PRIOR_PATH=SAVE_DIR/'G1_G2_Qwen_RAG_Full_Test.csv'
P2_GEN_PATH=SAVE_DIR/'G1_G2_G3_Qwen_RAG_Full_Test.csv'
prior=pd.read_csv(PRIOR_PATH) if PRIOR_PATH.exists() else pd.DataFrame()
prior_map={str(r['query']):r for _,r in prior.iterrows()} if len(prior) else {}

queries=test_df['query'].drop_duplicates().tolist()
if MAX_GENERATION_QUERIES is not None: queries=queries[:MAX_GENERATION_QUERIES]
if P2_GEN_PATH.exists():
    old=pd.read_csv(P2_GEN_PATH); done=set(old['query'].astype(str)); outrows=old.to_dict('records')
else: done=set(); outrows=[]
remaining=[q for q in queries if q not in done]
print('Target:',len(queries),'Remaining:',len(remaining),'Prior G1/G2 available:',len(prior_map)>0)

for q in tqdm(remaining,desc='G1/G2/G3 generation'):
    g=test_df[test_df['query']==q].copy()
    # G1 + G2: reuse previous answers when available
    if q in prior_map:
        pr=prior_map[q]
        g1_answer=pr.get('original_rag_answer'); g1_latency=float(pr.get('original_latency_s',np.nan)); g1_count=int(pr.get('original_context_count',0)); g1_context=pr.get('original_context','')
        g2_answer=pr.get('gated_rag_answer'); g2_latency=float(pr.get('gated_latency_s',np.nan)); g2_count=int(pr.get('gated_context_count',0)); g2_context=pr.get('gated_context',''); g2_abstain=int(pr.get('gate_abstained',0))
    else:
        b=g.sort_values('chunk_number').head(TOP_K); bctx=[('ORIGINAL',x) for x in b['chunk'].tolist()]
        g1_answer,g1_latency,_=generate_answer(q,bctx); g1_count=len(bctx); g1_context='\n\n'.join([x[1] for x in bctx])
        p1sel=g[g['evidence_confidence']>=P1_THRESHOLD].sort_values('evidence_confidence',ascending=False).head(TOP_K)
        p1ctx=[('P1_ACCEPT',x) for x in p1sel['chunk'].tolist()]
        if not p1ctx: g2_answer='ABSTAIN: No evidence passed the P1 confidence threshold.'; g2_latency=0.0; g2_abstain=1
        else: g2_answer,g2_latency,_=generate_answer(q,p1ctx); g2_abstain=0
        g2_count=len(p1ctx); g2_context='\n\n'.join([x[1] for x in p1ctx])
    # G3 P2
    p2sel=adaptive_select(g,P2_LOW,P2_HIGH,P2_TARGET)
    p2ctx=[(r['evidence_tier'],r['chunk']) for _,r in p2sel.iterrows()]
    if not p2ctx: g3_answer='ABSTAIN: No evidence passed the P2 adaptive gate.'; g3_latency=0.0; g3_abstain=1
    else: g3_answer,g3_latency,_=generate_answer(q,p2ctx); g3_abstain=0
    response=g['original_response'].dropna().astype(str).iloc[0] if ('original_response' in g.columns and g['original_response'].notna().any()) else None
    outrows.append({'query':q,'g1_context_count':g1_count,'g2_context_count':g2_count,'g3_context_count':len(p2ctx),'g1_answer':g1_answer,'g2_answer':g2_answer,'g3_answer':g3_answer,'g1_latency_s':g1_latency,'g2_latency_s':g2_latency,'g3_latency_s':g3_latency,'g2_abstained':g2_abstain,'g3_abstained':g3_abstain,'dataset_response':response,'g1_context':g1_context,'g2_context':g2_context,'g3_context':'\n\n'.join([x[1] for x in p2ctx])})
    pd.DataFrame(outrows).to_csv(P2_GEN_PATH,index=False)
generation_df=pd.DataFrame(outrows)
print('Generation complete:',P2_GEN_PATH)


## 9. End-to-end evaluation and final summary


In [ ]:
bi_encoder=SentenceTransformer(BI_ENCODER_NAME)
def split_sentences(t): return [x.strip() for x in re.split(r'(?<=[.!?])\s+',str(t).strip()) if x.strip()]
def citation_present(t): return int(bool(re.search(r'\[E\d+\]',str(t))))
def support_proxy(answer,context):
    ss=split_sentences(answer); cc=[x.strip() for x in str(context).split('\n\n') if x.strip()]
    if not ss or not cc: return np.nan
    a=bi_encoder.encode(ss,normalize_embeddings=True); c=bi_encoder.encode(cc,normalize_embeddings=True)
    return float((a@c.T).max(axis=1).mean())
def text_similarity(a,b):
    if pd.isna(a) or pd.isna(b): return np.nan
    a=str(a).strip(); b=str(b).strip()
    if not a or not b: return np.nan
    e=bi_encoder.encode([a,b],normalize_embeddings=True); return float(np.dot(e[0],e[1]))

ge=generation_df.copy()
for s in ['g1','g2','g3']:
    ge[f'{s}_citation']=ge[f'{s}_answer'].apply(citation_present)
    supports=[]; sims=[]
    for _,r in tqdm(ge.iterrows(),total=len(ge),desc=f'Evaluating {s}'):
        abstain=(s=='g2' and int(r['g2_abstained'])==1) or (s=='g3' and int(r['g3_abstained'])==1)
        supports.append(np.nan if abstain else support_proxy(r[f'{s}_answer'],r[f'{s}_context']))
        sims.append(np.nan if abstain else text_similarity(r[f'{s}_answer'],r['dataset_response']))
    ge[f'{s}_support_proxy']=supports; ge[f'{s}_response_similarity']=sims

END_SUMMARY=pd.DataFrame({
 'Metric':['Mean Context Count','Mean Latency (s)','Citation Presence Rate','Semantic Evidence-Support Proxy','Dataset-Response Similarity Proxy','Abstention Rate'],
 'G1 Original RAG':[ge['g1_context_count'].mean(),ge['g1_latency_s'].mean(),ge['g1_citation'].mean(),ge['g1_support_proxy'].mean(skipna=True),ge['g1_response_similarity'].mean(skipna=True),0.0],
 'G2 P1 Gate':[ge['g2_context_count'].mean(),ge['g2_latency_s'].mean(),ge['g2_citation'].mean(),ge['g2_support_proxy'].mean(skipna=True),ge['g2_response_similarity'].mean(skipna=True),ge['g2_abstained'].mean()],
 'G3 P2 Adaptive':[ge['g3_context_count'].mean(),ge['g3_latency_s'].mean(),ge['g3_citation'].mean(),ge['g3_support_proxy'].mean(skipna=True),ge['g3_response_similarity'].mean(skipna=True),ge['g3_abstained'].mean()]
})
display(END_SUMMARY.round(4))
ge.to_csv(SAVE_DIR/'G1_G2_G3_Qwen_RAG_Evaluated.csv',index=False)
END_SUMMARY.to_csv(SAVE_DIR/'G1_G2_G3_End_to_End_Summary.csv',index=False)

plot=END_SUMMARY[END_SUMMARY['Metric'].isin(['Citation Presence Rate','Semantic Evidence-Support Proxy','Dataset-Response Similarity Proxy'])].set_index('Metric')
ax=plot.plot(kind='bar',figsize=(10,5)); ax.set_ylim(0,1.02); ax.set_title('G1 Original vs G2 P1 vs G3 P2'); plt.xticks(rotation=10,ha='right'); plt.tight_layout(); plt.savefig(SAVE_DIR/'Figure_G1_G2_G3_End_to_End.png',dpi=220); plt.show()

summary='\n'.join([
 'P2 ADAPTIVE CONFIDENCE GATE — AUTOMATIC RESULT SUMMARY','='*72,
 f'P1 threshold: {P1_THRESHOLD:.2f}',
 f'P2 low/high/target: {P2_LOW:.2f} / {P2_HIGH:.2f} / {P2_TARGET}',
 '', 'TEST EVIDENCE COMPARISON', TEST_COMPARISON.round(4).to_string(index=False),
 '', 'END-TO-END QWEN COMPARISON', END_SUMMARY.round(4).to_string(index=False),
 '', 'NOTE: P2 thresholds were selected on validation only.',
 'NOTE: Dataset-response similarity is a proxy, not gold-answer accuracy.',
 'NOTE: Semantic evidence-support is an embedding proxy, not a true faithfulness metric.'
])
(SAVE_DIR/'P2_Adaptive_Gate_Automatic_Result_Summary.txt').write_text(summary,encoding='utf-8')
print(summary)
print('\nALL P2 WORK COMPLETE. Results saved in:',SAVE_DIR)


# Finished

Main new outputs:
- `P2_validation_grid.csv`
- `P2_Test_Evidence_Comparison.csv`
- `P2_Test_Query_Level_Results.csv`
- `P2_Selected_Test_Evidence.csv`
- `G1_G2_G3_Qwen_RAG_Full_Test.csv`
- `G1_G2_G3_Qwen_RAG_Evaluated.csv`
- `G1_G2_G3_End_to_End_Summary.csv`
- `P2_Adaptive_Gate_Automatic_Result_Summary.txt`
- `Figure_P2_Evidence_Comparison.png`
- `Figure_G1_G2_G3_End_to_End.png`
